
# Notebook 09 — Literature-search top-candidate search-query generation

**Project:** CMT Path A — Leakage-audited multi-ion computed insertion-electrode benchmark
**Notebook:** `_CMT_PUBLIC_NOTEBOOK_FILENAME_02_`

## Why this replacement notebook exists

The earlier Notebook 09 produced many exact Materials Project-style formula searches such as `Na1-3CoPCO7`, `Na3CoPCO7`, and `CoPCO7`. These are chemically meaningful inside Materials Project, but they are **not literature-search search strings**. Google Scholar often fails because papers normally write these compositions as grouped polyatomic formulas, for example:

- `Na3CoPCO7` → `Na3CoPO4CO3` or `Na3Co(CO3)(PO4)`
- `CoPCO7` → `CoPO4CO3` or `Co(CO3)(PO4)`
- `Na3FePCO7` → `Na3FePO4CO3` or `Na3Fe(CO3)(PO4)`
- `Na2FeCSO7` → `Na2FeCO3SO4` or `Na2Fe(CO3)(SO4)`

This notebook generates a **new manual literature-check CSV** where the first rows for each candidate are search-friendly formulas, grouped anion formulas, material names, and broader family terms.

## Important reviewer-safety rules

1. This notebook does **not scrape Google Scholar**.
2. It only creates reproducible search queries and blank manual-annotation tables.
3. Strict/unfriendly MP formula queries are archived in an audit file, not used as the main manual-check list.
4. Literature analogue evidence is not treated as validation of the Materials Project computed record.


In [ ]:

# ============================================================
# Cell 1 — Imports, configuration, and paths
# ============================================================
from __future__ import annotations

import os
import re
import json
import math
import platform
from pathlib import Path
from datetime import datetime, timezone, date
from urllib.parse import quote_plus

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

# -
# User-facing settings
# -
TOP_N_CANDIDATES = 30
RECOMMENDED_QUERIES_PER_CANDIDATE = 10
INCLUDE_ARCHIVED_UNFRIENDLY_QUERIES_IN_MANUAL_CSV = False

# If you already filled some rows, place the file here before running:
# provenance/supplementary/notebook_09/manual_inputs/09_manual_literature_annotations_FILLED.csv
MERGE_EXISTING_ANNOTATIONS_WHEN_QUERY_MATCHES = True

SEARCH_PLATFORM = "Google Scholar"
GOOGLE_SCHOLAR_URL = "https://scholar.google.com/scholar?q={query}"

# -
# Canonical clean-room input and output namespaces
# -
def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

NB12_DIR = artifact_namespace("08", REPOSITORY_ROOT)

BASE_DIR = artifact_namespace("09", REPOSITORY_ROOT)
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
MANUAL_INPUT_DIR = BASE_DIR / "manual_inputs"
LOG_DIR = BASE_DIR / "logs"

for d in [PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, MANUAL_INPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str, ensure_ascii=False),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "09_event_log.csv", index=False)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

def safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except Exception:
        pass
    return str(x)

print("Notebook 09 literature-search mode initialized")
print("Notebook 08 directory:", NB12_DIR if NB12_DIR else "not found yet; fallback candidate table will be attempted")
print("Notebook 09 outputs:", BASE_DIR)

log_event("init", "INFO", "Notebook 09 literature-search initialized", {"nb12_dir": str(NB12_DIR) if NB12_DIR else None})
save_event_log()


In [ ]:

# ============================================================
# Cell 2 — Load the top-30 sodium candidates
# ============================================================

def load_candidates_from_notebook_12(nb12_dir: Path) -> pd.DataFrame:
    top30_path = nb12_dir / "processed" / "08_sodium_top30_candidates_for_literature_review.csv"
    final_path = nb12_dir / "metadata" / "08_final_decision.json"
    if not top30_path.exists():
        raise FileNotFoundError(f"Missing {top30_path}")
    df = pd.read_csv(top30_path, low_memory=False)
    if final_path.exists():
        with open(final_path, "r", encoding="utf-8") as f:
            final_decision = json.load(f)
        print("Notebook 08 decision:", final_decision.get("final_decision"))
    return df


def find_fallback_manual_csv() -> Path | None:
    candidates = []
    search_dirs = [Path("."), BASE_DIR, MANUAL_INPUT_DIR]
    # /mnt/data fallback is useful in hosted/sandbox runs and harmless elsewhere.
    if Path("/mnt/data").exists():
        search_dirs.append(Path("/mnt/data"))
    patterns = [
        "09_manual_literature_annotations_REVIEWED*.csv",
        "09_manual_literature_annotations_FILLED*.csv",
        "09_manual_literature_annotations*.csv",
    ]
    for d in search_dirs:
        for pat in patterns:
            candidates.extend(sorted(d.glob(pat)))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        return None
    # Use newest modified file.
    return max(candidates, key=lambda p: p.stat().st_mtime)


def load_candidates_from_existing_manual_csv(path: Path) -> pd.DataFrame:
    m = pd.read_csv(path, low_memory=False)
    required = ["manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_discharge", "framework_formula", "coarse_family", "chemical_system_uid"]
    missing = [c for c in required if c not in m.columns]
    if missing:
        raise ValueError(f"Fallback manual CSV missing required columns: {missing}")
    out = (
        m[required]
        .drop_duplicates("manual_candidate_id")
        .sort_values("shortlist_priority_rank")
        .reset_index(drop=True)
    )
    return out

if NB12_DIR is not None:
    raw_candidates_df = load_candidates_from_notebook_12(NB12_DIR)
    input_source = "notebook_08_outputs"
else:
    fallback_csv = find_fallback_manual_csv()
    if fallback_csv is None:
        raise FileNotFoundError(
            "Could not find Notebook 08 outputs and no previous Notebook 09 manual CSV was found.\n"
            "Run Notebook 08 first, or place a previous 09_manual_literature_annotations*.csv file in this folder."
        )
    raw_candidates_df = load_candidates_from_existing_manual_csv(fallback_csv)
    input_source = f"fallback_existing_manual_csv:{fallback_csv}"
    print("Using fallback candidate list from:", fallback_csv)

required_cols = ["shortlist_priority_rank", "battery_formula", "formula_discharge", "framework_formula", "coarse_family"]
missing_cols = [c for c in required_cols if c not in raw_candidates_df.columns]
if missing_cols:
    raise ValueError(f"Candidate table missing required columns: {missing_cols}")

candidates_df = raw_candidates_df.copy()
candidates_df["shortlist_priority_rank"] = pd.to_numeric(candidates_df["shortlist_priority_rank"], errors="coerce")
# final_triage_score may exist in Notebook 08 but not in fallback CSV.
if "final_triage_score" not in candidates_df.columns:
    candidates_df["final_triage_score"] = 0.0

candidates_df = (
    candidates_df
    .sort_values(["shortlist_priority_rank", "final_triage_score"], ascending=[True, False])
    .head(TOP_N_CANDIDATES)
    .reset_index(drop=True)
)

if "manual_candidate_id" not in candidates_df.columns:
    candidates_df["manual_candidate_id"] = [f"Na_candidate_{i:02d}" for i in range(1, len(candidates_df) + 1)]
else:
    # Keep old IDs when they exist, otherwise regenerate.
    if candidates_df["manual_candidate_id"].isna().any() or (candidates_df["manual_candidate_id"].astype(str).str.strip() == "").any():
        candidates_df["manual_candidate_id"] = [f"Na_candidate_{i:02d}" for i in range(1, len(candidates_df) + 1)]

# Ensure optional columns exist.
for col in ["chemical_system_uid", "coarse_family", "battery_formula", "formula_discharge", "framework_formula"]:
    if col not in candidates_df.columns:
        candidates_df[col] = ""

candidates_df.to_csv(PROCESSED_DIR / "09_sorted_top30_candidates.csv", index=False)

input_audit = pd.DataFrame([{
    "input_source": input_source,
    "n_candidates_loaded": len(raw_candidates_df),
    "n_candidates_used": len(candidates_df),
    "missing_required_columns": "|".join(missing_cols),
}])
input_audit.to_csv(AUDIT_DIR / "09_input_coverage_audit.csv", index=False)

display(candidates_df[["manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_discharge", "framework_formula", "coarse_family", "chemical_system_uid"]].head(30))


In [ ]:

# ============================================================
# Cell 3 — Chemistry helpers for literature-search formulas
# ============================================================

ELEMENT_NAMES = {
    "Li": "lithium", "Na": "sodium", "K": "potassium", "Mg": "magnesium", "Ca": "calcium",
    "V": "vanadium", "Mn": "manganese", "Fe": "iron", "Co": "cobalt", "Ni": "nickel",
    "Cr": "chromium", "Ti": "titanium", "Cu": "copper", "Zn": "zinc", "Al": "aluminum",
    "Mo": "molybdenum", "W": "tungsten", "Nb": "niobium", "Zr": "zirconium", "Y": "yttrium",
    "Ag": "silver", "As": "arsenic", "Sb": "antimony", "Bi": "bismuth", "Sn": "tin",
    "P": "phosphorus", "S": "sulfur", "Si": "silicon", "C": "carbon", "O": "oxygen", "F": "fluorine", "H": "hydrogen",
}

POLYATOMIC_NAMES = {
    "PO4": "phosphate",
    "P2O7": "pyrophosphate",
    "CO3": "carbonate",
    "SO4": "sulfate",
    "AsO4": "arsenate",
    "SiO4": "silicate",
    "PO4F": "fluorophosphate",
}

ANION_ELEMENTS = {"O", "F", "P", "S", "C", "As", "Si", "H"}
ALKALI_ELEMENTS = {"Li", "Na", "K", "Mg", "Ca"}


def clean_query_text(q: str) -> str:
    q = re.sub(r"\s+", " ", safe_str(q)).strip()
    return q


def normalize_query(q: str) -> str:
    q = safe_str(q).lower().strip()
    q = q.replace("–", "-").replace("−", "-")
    q = re.sub(r"\s+", " ", q)
    return q


def normalize_formula_token(s: str) -> str:
    return safe_str(s).strip().replace(" ", "").replace("·", "")


def _add_counts(a: dict, b: dict, mult: float = 1.0) -> dict:
    out = dict(a)
    for k, v in b.items():
        out[k] = out.get(k, 0.0) + v * mult
    return out


def parse_formula_counts(formula: str) -> dict:
    """Small formula parser supporting element symbols, decimals, and parentheses.
    It is intentionally conservative and used only for search-query generation.
    """
    s = normalize_formula_token(formula)
    # Battery-window formulas like Na1-3Co... are not parsed as formulas; use discharged/framework formulas instead.
    if "-" in s:
        return {}
    i = 0
    n = len(s)

    def parse_number() -> float:
        nonlocal i
        start = i
        while i < n and (s[i].isdigit() or s[i] == "."):
            i += 1
        if start == i:
            return 1.0
        try:
            return float(s[start:i])
        except Exception:
            return 1.0

    def parse_group(stop_char=None) -> dict:
        nonlocal i
        counts = {}
        while i < n:
            ch = s[i]
            if stop_char and ch == stop_char:
                i += 1
                break
            if ch == "(":
                i += 1
                inner = parse_group(")")
                mult = parse_number()
                counts = _add_counts(counts, inner, mult)
            elif ch.isupper():
                elem = ch
                i += 1
                if i < n and s[i].islower():
                    elem += s[i]
                    i += 1
                num = parse_number()
                counts[elem] = counts.get(elem, 0.0) + num
            else:
                # Unknown character, skip safely.
                i += 1
        return counts

    return parse_group()


def count_to_string(x: float) -> str:
    if abs(x - 1.0) < 1e-8:
        return ""
    if abs(x - round(x)) < 1e-8:
        return str(int(round(x)))
    return (f"{x:g}").rstrip("0").rstrip(".")


def formula_from_ordered_counts(order: list[str], counts: dict) -> str:
    parts = []
    for e in order:
        v = counts.get(e, 0)
        if v and abs(v) > 1e-8:
            parts.append(f"{e}{count_to_string(v)}")
    for e in sorted(counts):
        if e not in order and counts.get(e, 0) and abs(counts[e]) > 1e-8:
            parts.append(f"{e}{count_to_string(counts[e])}")
    return "".join(parts)


def likely_transition_metals(counts: dict) -> list[str]:
    metals = []
    for e in counts:
        if e not in ANION_ELEMENTS and e not in ALKALI_ELEMENTS and e in ELEMENT_NAMES:
            metals.append(e)
    return sorted(metals)


def element_phrase(elements: list[str]) -> str:
    names = [ELEMENT_NAMES.get(e, e).lower() for e in elements if e]
    if not names:
        return ""
    if len(names) == 1:
        return names[0]
    return " ".join(names)


def contains_close(formula: str, pattern: str) -> bool:
    return pattern.lower() in safe_str(formula).lower().replace(" ", "")


def grouped_polyoxo_variants(formula: str) -> list[str]:
    """Return literature-search variants for condensed formulas.
    Example: Na3CoPCO7 -> Na3CoPO4CO3 and Na3Co(CO3)(PO4).
    """
    formula = normalize_formula_token(formula)
    counts = parse_formula_counts(formula)
    if not counts:
        return [formula] if formula else []

    variants = []
    na = counts.get("Na", 0)
    metals = likely_transition_metals(counts)
    metal_part = "".join(f"{m}{count_to_string(counts.get(m, 0))}" for m in metals)
    prefix = ("Na" + count_to_string(na)) if na else ""

    # Extract known polyanion counts.
    c = counts.get("C", 0)
    p = counts.get("P", 0)
    s = counts.get("S", 0)
    ars = counts.get("As", 0)
    si = counts.get("Si", 0)
    f = counts.get("F", 0)
    o = counts.get("O", 0)
    h = counts.get("H", 0)

    def group_token(group: str, n: float) -> str:
        # Literature searches are more readable and more robust with explicit polyatomic groups.
        if abs(n - 1) < 1e-8:
            return f"({group})"
        return f"({group}){count_to_string(n)}"

    def compact_token(base: str, n: float) -> str:
        # Avoid invalid strings such as PO42. For repeated polyatomic groups, use parentheses.
        if abs(n - 1) < 1e-8:
            return base
        return f"({base}){count_to_string(n)}"

    # Carbonate + phosphate/carbonophosphate: PCO7 = PO4 + CO3.
    if c > 0 and p > 0 and abs(o - (3*c + 4*p)) < 1e-8:
        compact = prefix + metal_part + compact_token("PO4", p) + compact_token("CO3", c)
        grouped1 = prefix + metal_part + group_token("CO3", c) + group_token("PO4", p)
        grouped2 = prefix + metal_part + group_token("PO4", p) + group_token("CO3", c)
        variants.extend([compact, grouped1, grouped2])

    # Carbonate + sulfate: CSO7 = CO3 + SO4.
    if c > 0 and s > 0 and abs(o - (3*c + 4*s)) < 1e-8:
        compact = prefix + metal_part + compact_token("CO3", c) + compact_token("SO4", s)
        grouped1 = prefix + metal_part + group_token("CO3", c) + group_token("SO4", s)
        grouped2 = prefix + metal_part + group_token("SO4", s) + group_token("CO3", c)
        variants.extend([compact, grouped1, grouped2])

    # Carbonate + arsenate: AsCO7 = AsO4 + CO3.
    if c > 0 and ars > 0 and abs(o - (3*c + 4*ars)) < 1e-8:
        compact = prefix + metal_part + compact_token("AsO4", ars) + compact_token("CO3", c)
        grouped1 = prefix + metal_part + group_token("CO3", c) + group_token("AsO4", ars)
        grouped2 = prefix + metal_part + group_token("AsO4", ars) + group_token("CO3", c)
        variants.extend([compact, grouped1, grouped2])

    # Carbonate + silicate: SiCO7 = SiO4 + CO3.
    if c > 0 and si > 0 and abs(o - (3*c + 4*si)) < 1e-8:
        compact = prefix + metal_part + compact_token("SiO4", si) + compact_token("CO3", c)
        grouped1 = prefix + metal_part + group_token("CO3", c) + group_token("SiO4", si)
        grouped2 = prefix + metal_part + group_token("SiO4", si) + group_token("CO3", c)
        variants.extend([compact, grouped1, grouped2])

    # Normal phosphate/pyrophosphate/fluorophosphate formulas are already literature-search.
    if not variants:
        variants.append(formula)

    # Ensure original is present but not necessarily first.
    if formula not in variants:
        variants.append(formula)

    # Deduplicate while preserving order.
    out = []
    seen = set()
    for v in variants:
        v = normalize_formula_token(v)
        if v and v not in seen:
            out.append(v)
            seen.add(v)
    return out


def variable_sodium_variants(formula: str) -> list[str]:
    variants = []
    for v in grouped_polyoxo_variants(formula):
        # Replace initial Na<number> with Nax. If no Na, prepend nothing.
        if re.match(r"^Na[0-9.]+", v):
            variants.append(re.sub(r"^Na[0-9.]+", "Nax", v))
            # Some papers use Na_x or Na3-x style; simple query uses text Nax.
        elif v.startswith("Na"):
            variants.append("Nax" + v[2:])
    return list(dict.fromkeys(variants))


def formula_is_literature_unfriendly(formula: str) -> bool:
    s = normalize_formula_token(formula)
    if not s:
        return True
    if "-" in s:
        return True
    # Condensed patterns that should be expanded for search.
    bad_patterns = ["PCO7", "CSO7", "AsCO7", "SiCO7", "CO7"]
    return any(p.lower() in s.lower() for p in bad_patterns)

# Quick sanity check for the troublesome formulas from Candidate 6 and related rows.
for test_formula in ["Na3CoPCO7", "CoPCO7", "Na2FeCSO7", "Na3FeAsCO7", "Na6Fe2C4SO16", "Na5Mn2P2(CO7)2"]:
    print(test_formula, "->", grouped_polyoxo_variants(test_formula)[:4])


In [ ]:

# ============================================================
# Cell 4 — Literature-search material phrase helpers
# ============================================================

def phrase_variants_for_candidate(cand: dict) -> list[tuple[str, str]]:
    """Return (query_type, phrase) pairs for material-name and family searches."""
    discharge_formula = safe_str(cand.get("formula_discharge", ""))
    framework_formula = safe_str(cand.get("framework_formula", ""))
    family = safe_str(cand.get("coarse_family", "")).lower()

    counts = parse_formula_counts(discharge_formula) or parse_formula_counts(framework_formula)
    metals = likely_transition_metals(counts)
    metal_name = element_phrase(metals)
    phrases = []

    # Formula-specific polyanion language.
    c = counts.get("C", 0)
    p = counts.get("P", 0)
    s = counts.get("S", 0)
    ars = counts.get("As", 0)
    si = counts.get("Si", 0)
    f = counts.get("F", 0)
    o = counts.get("O", 0)

    if metal_name:
        if c > 0 and p > 0:
            phrases.append(("material_phrase_carbonophosphate", f"sodium {metal_name} carbonophosphate cathode"))
            phrases.append(("material_phrase_carbonate_phosphate", f"sodium {metal_name} carbonate phosphate cathode"))
            phrases.append(("family_formula_carbonophosphate", f"Na3MPO4CO3 sodium ion battery cathode {metal_name}"))
        if c > 0 and s > 0:
            phrases.append(("material_phrase_sulfate_carbonate", f"sodium {metal_name} sulfate carbonate cathode"))
            phrases.append(("material_phrase_carbonate_sulfate", f"sodium {metal_name} carbonate sulfate cathode"))
        if c > 0 and ars > 0:
            phrases.append(("material_phrase_arsenate_carbonate", f"sodium {metal_name} arsenate carbonate cathode"))
            phrases.append(("material_phrase_carbonate_arsenate", f"sodium {metal_name} carbonate arsenate cathode"))
        if c > 0 and si > 0:
            phrases.append(("material_phrase_silicate_carbonate", f"sodium {metal_name} silicate carbonate cathode"))
            phrases.append(("material_phrase_carbonate_silicate", f"sodium {metal_name} carbonate silicate cathode"))
        if p > 0 and f > 0:
            phrases.append(("material_phrase_fluorophosphate", f"sodium {metal_name} fluorophosphate cathode"))
        if p > 0 and o > 0 and abs(o - 3.5*p) < 1e-8:
            phrases.append(("material_phrase_pyrophosphate", f"sodium {metal_name} pyrophosphate cathode"))
        if p > 0 and "phosphate" in family:
            phrases.append(("material_phrase_phosphate", f"sodium {metal_name} phosphate cathode"))
        if s > 0 and "sulf" in family:
            phrases.append(("material_phrase_sulfate", f"sodium {metal_name} sulfate cathode"))
        if f > 0 and "fluoride" in family:
            phrases.append(("material_phrase_fluoride", f"{metal_name} fluoride sodium ion battery cathode"))
        if "oxide" in family or (o > 0 and not (p or s or c or ars or si or f)):
            phrases.append(("material_phrase_oxide", f"sodium {metal_name} oxide cathode"))
            phrases.append(("layered_oxide_phrase", f"Nax{''.join(metals)}O2 sodium ion battery cathode" if metals else "layered oxide sodium ion battery cathode"))

    # Broad family fallbacks.
    if "phosphate" in family:
        phrases.append(("broad_family_polyanion", "polyanion phosphate sodium ion battery cathode"))
    if "sulf" in family:
        phrases.append(("broad_family_sulfate", "sulfate polyanion sodium ion battery cathode"))
    if "fluoride" in family:
        phrases.append(("broad_family_fluoride", "transition metal fluoride sodium ion battery cathode"))
    if "oxide" in family:
        phrases.append(("broad_family_oxide", "layered transition metal oxide sodium ion battery cathode"))
    if "silicate" in family:
        phrases.append(("broad_family_silicate", "sodium transition metal silicate cathode"))

    # Deduplicate.
    out, seen = [], set()
    for qt, ph in phrases:
        ph = clean_query_text(ph)
        key = ph.lower()
        if ph and key not in seen:
            out.append((qt, ph))
            seen.add(key)
    return out


def make_query_url(query: str) -> str:
    return GOOGLE_SCHOLAR_URL.format(query=quote_plus(query))


def add_query(rows: list, cand: dict, query_type: str, query: str, priority: int, note: str, tier: str = "A_try_first", manual_required: bool = True):
    query = clean_query_text(query)
    if not query:
        return
    rows.append({
        "manual_candidate_id": cand.get("manual_candidate_id"),
        "shortlist_priority_rank": cand.get("shortlist_priority_rank"),
        "battery_formula": cand.get("battery_formula"),
        "formula_discharge": cand.get("formula_discharge"),
        "framework_formula": cand.get("framework_formula"),
        "coarse_family": cand.get("coarse_family"),
        "chemical_system_uid": cand.get("chemical_system_uid", ""),
        "query_type": query_type,
        "query_priority": priority,
        "query_priority_tier": tier,
        "search_platform": SEARCH_PLATFORM,
        "search_query": query,
        "search_url": make_query_url(query),
        "query_note": note,
        "manual_check_required": bool(manual_required),
        "recommended_action": "search_this_first" if tier == "A_try_first" else ("search_if_needed" if manual_required else "do_not_search_main_csv"),
    })


In [ ]:

# ============================================================
# Cell 5 — Generate literature-search query table
# ============================================================

query_rows = []
archived_rows = []

for _, row in candidates_df.iterrows():
    cand = row.to_dict()
    battery_formula = safe_str(cand.get("battery_formula", "")).strip()
    discharge_formula = safe_str(cand.get("formula_discharge", "")).strip()
    framework_formula = safe_str(cand.get("framework_formula", "")).strip()

    discharge_variants = grouped_polyoxo_variants(discharge_formula)
    framework_variants = grouped_polyoxo_variants(framework_formula)
    variable_variants = variable_sodium_variants(discharge_formula)

    priority = 1

    # A. Literature-search exact/expanded discharged formulas.
    for i, f in enumerate(discharge_variants[:4]):
        if not f or formula_is_literature_unfriendly(f):
            continue
        add_query(query_rows, cand, "literature_friendly_exact_formula_sodium_ion", f"{f} sodium ion battery cathode", priority,
                  "Search-friendly discharged formula. Uses grouped/expanded anion notation when needed; no strict quotes to tolerate formatting differences.", "A_try_first", True)
        priority += 1
        add_query(query_rows, cand, "literature_friendly_exact_formula_electrode", f"{f} electrode cathode", priority,
                  "Search-friendly discharged formula with electrode/cathode words.", "A_try_first", True)
        priority += 1

    # B. Quoted exact formula only after no-quote version.
    for f in discharge_variants[:2]:
        if not f or formula_is_literature_unfriendly(f):
            continue
        add_query(query_rows, cand, "quoted_literature_formula", f'"{f}" sodium ion battery cathode', priority,
                  "Quoted formula query. Use after the unquoted query; quotes can miss papers using subscripts/spaces.", "B_try_second", True)
        priority += 1

    # C. Variable sodium formula variants.
    for f in variable_variants[:3]:
        if not f or formula_is_literature_unfriendly(f):
            continue
        add_query(query_rows, cand, "variable_sodium_literature_formula", f"{f} sodium ion battery cathode", priority,
                  "Variable-sodium literature formula. Useful when papers report Nax phases instead of a fully sodiated formula.", "B_try_second", True)
        priority += 1

    # D. Framework formula variants.
    for f in framework_variants[:3]:
        if not f or formula_is_literature_unfriendly(f):
            continue
        add_query(query_rows, cand, "literature_friendly_framework_formula", f"{f} sodium ion battery cathode", priority,
                  "Framework-level search-friendly formula. This is analogue evidence, not exact formula validation.", "B_try_second", True)
        priority += 1

    # E. Material phrase and family phrase searches.
    for qt, phrase in phrase_variants_for_candidate(cand)[:8]:
        add_query(query_rows, cand, qt, phrase, priority,
                  "Material-name or family-name query. Use to find papers that do not write the exact formula in the title/abstract.", "C_broad_fallback", True)
        priority += 1

    # F. Archive literature-unfriendly old-style queries for audit only.
    strict_candidates = [battery_formula, discharge_formula, framework_formula]
    for f in strict_candidates:
        if f and formula_is_literature_unfriendly(f):
            archived_rows.append({
                "manual_candidate_id": cand.get("manual_candidate_id"),
                "shortlist_priority_rank": cand.get("shortlist_priority_rank"),
                "formula": f,
                "reason_not_literature_friendly": "MP battery-window notation or condensed polyoxo formula; use grouped/expanded formula variants instead.",
                "suggested_replacements": " | ".join(grouped_polyoxo_variants(f)),
            })
            if INCLUDE_ARCHIVED_UNFRIENDLY_QUERIES_IN_MANUAL_CSV:
                add_query(query_rows, cand, "archived_unfriendly_formula_do_not_prioritize", f'"{f}" sodium ion battery cathode', priority,
                          "Archived only. This query is not literature-search and should not be used as the main search evidence.", "D_archived_not_recommended", False)
                priority += 1

query_table_df = pd.DataFrame(query_rows)

# Deduplicate candidate-query pairs while preserving the best priority.
query_table_df["normalized_search_query"] = query_table_df["search_query"].map(normalize_query)
query_table_df = (
    query_table_df
    .sort_values(["shortlist_priority_rank", "query_priority"])
    .drop_duplicates(["manual_candidate_id", "normalized_search_query"], keep="first")
    .reset_index(drop=True)
)

# Keep a manageable number of recommended queries per candidate.
manual_required_df = query_table_df[query_table_df["manual_check_required"]].copy()
manual_required_df = (
    manual_required_df
    .sort_values(["shortlist_priority_rank", "query_priority"])
    .groupby("manual_candidate_id", as_index=False, group_keys=False)
    .head(RECOMMENDED_QUERIES_PER_CANDIDATE)
    .reset_index(drop=True)
)

archived_df = pd.DataFrame(archived_rows).drop_duplicates() if archived_rows else pd.DataFrame(columns=["manual_candidate_id", "shortlist_priority_rank", "formula", "reason_not_literature_friendly", "suggested_replacements"])

# Reassign reproducible query IDs after filtering.
manual_required_df = manual_required_df.sort_values(["shortlist_priority_rank", "query_priority"]).reset_index(drop=True)
manual_required_df.insert(0, "query_id", [f"Q{i:04d}" for i in range(1, len(manual_required_df) + 1)])

# Save tables.
query_table_df.to_csv(PROCESSED_DIR / "09_literature_friendly_query_table_all_alternatives.csv", index=False)
manual_required_df.to_csv(PROCESSED_DIR / "09_literature_friendly_query_table_recommended.csv", index=False)
archived_df.to_csv(AUDIT_DIR / "09_literature_unfriendly_query_audit.csv", index=False)

query_generation_audit = (
    manual_required_df
    .groupby(["manual_candidate_id", "shortlist_priority_rank"], dropna=False)
    .agg(
        n_recommended_queries=("search_query", "count"),
        n_try_first=("query_priority_tier", lambda s: int((s == "A_try_first").sum())),
        n_try_second=("query_priority_tier", lambda s: int((s == "B_try_second").sum())),
        n_broad_fallback=("query_priority_tier", lambda s: int((s == "C_broad_fallback").sum())),
    )
    .reset_index()
)
query_generation_audit["status"] = np.where(query_generation_audit["n_recommended_queries"] >= 6, "PASS", "CHECK")
query_generation_audit.to_csv(AUDIT_DIR / "09_query_generation_coverage_audit.csv", index=False)

print("Recommended manual-search rows:", len(manual_required_df))
print("Archived unfriendly formulas:", len(archived_df))
display(manual_required_df[["query_id", "manual_candidate_id", "shortlist_priority_rank", "query_type", "query_priority_tier", "search_query", "query_note"]].head(40))


In [ ]:

# ============================================================
# Cell 6 — Create blank manual annotation CSV from literature-search queries
# ============================================================

ANNOTATION_STATUS_ALLOWED = [
    "not_checked",
    "exact_or_near_exact_formula",
    "same_framework_analogue",
    "same_family_same_transition_metal",
    "same_family_analogue",
    "general_background_only",
    "no_relevant_literature_found",
    "unclear_needs_second_check",
]

ANALOGUE_LEVEL_ALLOWED = [
    "not_checked",
    "exact_formula",
    "near_exact_formula",
    "same_framework",
    "same_family_same_transition_metal",
    "same_family",
    "background_only",
    "none",
    "unclear",
]

COMPUTED_OR_EXPERIMENTAL_ALLOWED = [
    "not_checked",
    "experimental",
    "computed",
    "review",
    "mixed",
    "none",
    "unclear",
]

manual_template_df = manual_required_df.copy()
manual_template_df["manual_checked"] = False
manual_template_df["annotation_status"] = "not_checked"
manual_template_df["analogue_level"] = "not_checked"
manual_template_df["relevance_score_0_to_3"] = ""
manual_template_df["citation_title"] = ""
manual_template_df["citation_year"] = ""
manual_template_df["doi"] = ""
manual_template_df["url"] = ""
manual_template_df["source_journal_or_database"] = ""
manual_template_df["same_sodium_working_ion"] = ""
manual_template_df["same_formula"] = ""
manual_template_df["same_framework"] = ""
manual_template_df["same_transition_metal"] = ""
manual_template_df["same_anion_framework"] = ""
manual_template_df["electrochemical_evidence_reported"] = ""
manual_template_df["computed_or_experimental"] = "not_checked"
manual_template_df["manual_evidence_note"] = ""
manual_template_df["reviewer_initials"] = ""
manual_template_df["manual_check_date"] = ""
manual_template_df["needs_second_check"] = ""
manual_template_df["evidence_confidence_0_to_3"] = ""
manual_template_df["do_not_use_as_discovery_claim"] = True

# Extra helper columns for the manual reviewer.
manual_template_df["manual_filling_hint"] = (
    "Prefer primary experimental sodium-ion cathode papers. If none, accept exact/near-exact computed papers. "
    "Use reviews only as background. Never claim this validates the MP record."
)
manual_template_df["negative_result_rule"] = (
    "Only mark no_relevant_literature_found for this exact query after checking it. "
    "Do not conclude the whole candidate has no literature unless several A/B/C queries fail."
)

blank_copy_path = PROCESSED_DIR / "09_manual_literature_annotations_BLANK_.csv"
manual_template_df.to_csv(blank_copy_path, index=False)

# Also copy to manual_inputs as a convenient file for the user to fill later.
manual_input_blank_path = MANUAL_INPUT_DIR / "09_manual_literature_annotations_BLANK_.csv"
manual_template_df.to_csv(manual_input_blank_path, index=False)

print("Blank literature-search manual CSV saved:")
print(" -", blank_copy_path)
print(" -", manual_input_blank_path)
display(manual_template_df[["query_id", "manual_candidate_id", "search_query", "query_priority_tier", "manual_checked", "annotation_status"]].head(20))


In [ ]:
# ============================================================
# Cell 7 — Optional carry-over from existing filled annotation CSVs
#  VERSION: fixes pandas Arrow string dtype assignment error
# ============================================================

def find_existing_filled_annotation_csvs() -> list[Path]:
    candidates = []
    search_dirs = [Path("."), MANUAL_INPUT_DIR, PROCESSED_DIR]

    # Useful if running inside a sandbox or copied folder
    if Path("/mnt/data").exists():
        search_dirs.append(Path("/mnt/data"))

    patterns = [
        "09_manual_literature_annotations_FILLED*.csv",
        "09_manual_literature_annotations_REVIEWED*.csv",
    ]

    for d in search_dirs:
        for pat in patterns:
            candidates.extend(sorted(d.glob(pat)))

    # Exclude the new blank/literature-search outputs to avoid self-copying
    candidates = [
        p for p in candidates
        if p.is_file()
        and "BLANK" not in p.name.upper()
        and "" not in p.name.upper()
    ]

    # Newest first
    return sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)


# IMPORTANT FIX:
# Force object dtype so pandas does not use ArrowStringArray during assignment.
manual_with_carryover_df = manual_template_df.copy().astype("object")
carryover_audit_rows = []

if MERGE_EXISTING_ANNOTATIONS_WHEN_QUERY_MATCHES:
    existing_files = find_existing_filled_annotation_csvs()
else:
    existing_files = []

if existing_files:
    print("Existing annotation files found for possible carry-over:")
    for p in existing_files[:5]:
        print(" -", p)

    old_frames = []

    for p in existing_files:
        try:
            # IMPORTANT FIX:
            # Read all old CSV columns as string/object.
            tmp = pd.read_csv(
                p,
                dtype=str,
                keep_default_na=False,
                low_memory=False,
                encoding="utf-8-sig"
            )
            tmp["_source_file"] = str(p)
            old_frames.append(tmp)
        except UnicodeDecodeError:
            try:
                tmp = pd.read_csv(
                    p,
                    dtype=str,
                    keep_default_na=False,
                    low_memory=False,
                    encoding="cp1252"
                )
                tmp["_source_file"] = str(p)
                old_frames.append(tmp)
            except Exception as e:
                log_event(
                    "carryover",
                    "WARNING",
                    f"Could not read old annotation file: {p}",
                    {"error": str(e)}
                )
        except Exception as e:
            log_event(
                "carryover",
                "WARNING",
                f"Could not read old annotation file: {p}",
                {"error": str(e)}
            )

    if old_frames:
        old = pd.concat(old_frames, ignore_index=True, sort=False).astype("object")

        required_old_cols = ["manual_checked", "search_query", "manual_candidate_id"]

        if all(c in old.columns for c in required_old_cols):
            old["_norm_query"] = old["search_query"].map(normalize_query)
            old["_checked_bool"] = (
                old["manual_checked"]
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "1", "yes", "y"])
            )

            old_checked = old[old["_checked_bool"]].copy()

            # Keep newest checked row for candidate + exact query match
            old_checked = old_checked.drop_duplicates(
                ["manual_candidate_id", "_norm_query"],
                keep="first"
            )

            manual_with_carryover_df["_norm_query"] = manual_with_carryover_df["search_query"].map(normalize_query)

            key_cols = ["manual_candidate_id", "_norm_query"]

            fill_cols = [
                "manual_checked",
                "annotation_status",
                "analogue_level",
                "relevance_score_0_to_3",
                "citation_title",
                "citation_year",
                "doi",
                "url",
                "source_journal_or_database",
                "same_sodium_working_ion",
                "same_formula",
                "same_framework",
                "same_transition_metal",
                "same_anion_framework",
                "electrochemical_evidence_reported",
                "computed_or_experimental",
                "manual_evidence_note",
                "reviewer_initials",
                "manual_check_date",
                "needs_second_check",
                "evidence_confidence_0_to_3",
                "do_not_use_as_discovery_claim",
            ]

            available_fill_cols = [
                c for c in fill_cols
                if c in old_checked.columns and c in manual_with_carryover_df.columns
            ]

            old_small = old_checked[key_cols + available_fill_cols + ["_source_file"]].copy().astype("object")

            merged = manual_with_carryover_df.merge(
                old_small,
                on=key_cols,
                how="left",
                suffixes=("", "_old")
            ).astype("object")

            # IMPORTANT FIX:
            # Use .to_numpy(dtype=object), not direct Series assignment.
            # This avoids Arrow string dtype errors.
            for c in available_fill_cols:
                old_col = c + "_old"
                if old_col in merged.columns:
                    mask = merged["_source_file"].notna()

                    if mask.any():
                        merged.loc[mask, c] = merged.loc[mask, old_col].astype("object").to_numpy()

                    merged = merged.drop(columns=[old_col])

            if "_source_file" in merged.columns:
                n_carry = int(merged["_source_file"].notna().sum())

                carryover_audit_rows = (
                    merged.loc[
                        merged["_source_file"].notna(),
                        ["query_id", "manual_candidate_id", "search_query", "_source_file"]
                    ]
                    .copy()
                    .to_dict("records")
                )

                merged = merged.drop(columns=["_source_file"])
            else:
                n_carry = 0

            manual_with_carryover_df = merged.drop(
                columns=[c for c in ["_norm_query"] if c in merged.columns]
            ).astype("object")

            print("Rows carried over by exact candidate+query match:", n_carry)

        else:
            print("Existing files did not have required carry-over columns; no carry-over applied.")
            print("Required columns:", required_old_cols)
    else:
        print("No readable old annotation files were found; no carry-over applied.")

else:
    print("No existing filled/reviewed annotation CSV found for carry-over. Blank file only.")


carryover_path = PROCESSED_DIR / "09_manual_literature_annotations_WITH_CARRYOVER.csv"
manual_with_carryover_df.to_csv(carryover_path, index=False, encoding="utf-8-sig")

carryover_audit_path = AUDIT_DIR / "09_annotation_carryover_audit.csv"
pd.DataFrame(carryover_audit_rows).to_csv(carryover_audit_path, index=False, encoding="utf-8-sig")

print("Carry-over manual CSV saved:", carryover_path)
print("Carry-over audit saved:", carryover_audit_path)

display(
    manual_with_carryover_df[
        ["query_id", "manual_candidate_id", "search_query", "manual_checked", "annotation_status"]
    ].head(30)
)

In [ ]:

# ============================================================
# Cell 8 — Validate filled/carry-over annotations and summarize progress
# ============================================================

manual_df = manual_with_carryover_df.copy()

# Normalize booleans.
def to_bool_series(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

manual_df["row_is_checked"] = to_bool_series(manual_df["manual_checked"])
checked_df = manual_df[manual_df["row_is_checked"]].copy()

validation_issues = []

if len(checked_df):
    bad_status = checked_df[~checked_df["annotation_status"].isin(ANNOTATION_STATUS_ALLOWED)]
    if len(bad_status):
        validation_issues.append({"issue": "invalid_annotation_status", "n_rows": len(bad_status), "examples": "|".join(sorted(bad_status["annotation_status"].astype(str).unique())[:10])})

    bad_level = checked_df[~checked_df["analogue_level"].isin(ANALOGUE_LEVEL_ALLOWED)]
    if len(bad_level):
        validation_issues.append({"issue": "invalid_analogue_level", "n_rows": len(bad_level), "examples": "|".join(sorted(bad_level["analogue_level"].astype(str).unique())[:10])})

    bad_compute = checked_df[~checked_df["computed_or_experimental"].isin(COMPUTED_OR_EXPERIMENTAL_ALLOWED)]
    if len(bad_compute):
        validation_issues.append({"issue": "invalid_computed_or_experimental", "n_rows": len(bad_compute), "examples": "|".join(sorted(bad_compute["computed_or_experimental"].astype(str).unique())[:10])})

    score_num = pd.to_numeric(checked_df["relevance_score_0_to_3"], errors="coerce")
    bad_score = checked_df[score_num.isna() | (score_num < 0) | (score_num > 3)]
    if len(bad_score):
        validation_issues.append({"issue": "invalid_relevance_score_0_to_3", "n_rows": len(bad_score), "examples": "|".join(bad_score["query_id"].astype(str).head(10))})

validation_issues_df = pd.DataFrame(validation_issues)
validation_issues_df.to_csv(AUDIT_DIR / "09_manual_annotation_validation_issues.csv", index=False)

progress_by_candidate = (
    manual_df
    .groupby(["manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_discharge", "framework_formula"], dropna=False)
    .agg(
        n_queries=("search_query", "count"),
        n_checked=("row_is_checked", "sum"),
        n_try_first=("query_priority_tier", lambda s: int((s == "A_try_first").sum())),
        n_checked_positive=("annotation_status", lambda s: int(s.isin(["exact_or_near_exact_formula", "same_framework_analogue", "same_family_same_transition_metal", "same_family_analogue"]).sum())),
        n_checked_negative=("annotation_status", lambda s: int((s == "no_relevant_literature_found").sum())),
    )
    .reset_index()
)
progress_by_candidate["candidate_manual_status"] = np.where(
    progress_by_candidate["n_checked"] >= 2,
    "started_or_sufficient",
    "needs_manual_check",
)
progress_by_candidate.to_csv(AUDIT_DIR / "09_manual_review_progress_by_candidate.csv", index=False)

print("Checked rows:", len(checked_df), "/", len(manual_df))
print("Validation issues:", len(validation_issues_df))
display(progress_by_candidate.head(30))
if len(validation_issues_df):
    display(validation_issues_df)


In [ ]:

# ============================================================
# Cell 9 — Build candidate-level literature analogue summary
# ============================================================

summary_rows = []
for _, g in manual_df.groupby("manual_candidate_id", dropna=False):
    g = g.copy()
    checked = g[g["row_is_checked"]].copy()
    if len(checked):
        checked["relevance_score_numeric"] = pd.to_numeric(checked["relevance_score_0_to_3"], errors="coerce").fillna(-1)
        checked["confidence_numeric"] = pd.to_numeric(checked["evidence_confidence_0_to_3"], errors="coerce").fillna(-1)
        best = checked.sort_values(["relevance_score_numeric", "confidence_numeric", "query_priority"], ascending=[False, False, True]).iloc[0]
        best_status = best.get("annotation_status", "not_checked")
        best_level = best.get("analogue_level", "not_checked")
        best_score = best.get("relevance_score_numeric", -1)
        best_title = best.get("citation_title", "")
        best_url = best.get("url", "")
        best_note = best.get("manual_evidence_note", "")
    else:
        best_status = "not_checked"
        best_level = "not_checked"
        best_score = np.nan
        best_title = ""
        best_url = ""
        best_note = ""

    first = g.iloc[0]
    summary_rows.append({
        "manual_candidate_id": first.get("manual_candidate_id"),
        "shortlist_priority_rank": first.get("shortlist_priority_rank"),
        "battery_formula": first.get("battery_formula"),
        "formula_discharge": first.get("formula_discharge"),
        "framework_formula": first.get("framework_formula"),
        "coarse_family": first.get("coarse_family"),
        "n_queries_generated": len(g),
        "n_checked_rows": int(g["row_is_checked"].sum()),
        "best_annotation_status": best_status,
        "best_analogue_level": best_level,
        "best_relevance_score_0_to_3": best_score,
        "best_citation_title": best_title,
        "best_url": best_url,
        "safe_interpretation": (
            "Literature analogue support only. Do not claim validation of the Materials Project computed record or triage rank. "
            + safe_str(best_note)[:600]
        ).strip(),
    })

candidate_summary_df = pd.DataFrame(summary_rows).sort_values("shortlist_priority_rank")
candidate_summary_df.to_csv(PROCESSED_DIR / "09_candidate_literature_analogue_summary.csv", index=False)

display(candidate_summary_df.head(30))


In [ ]:

# ============================================================
# Cell 10 — Reviewer-safety audit
# ============================================================

unfriendly_in_manual = manual_required_df[manual_required_df["search_query"].map(lambda q: any(p in normalize_query(q) for p in ["pcO7".lower(), "cso7", "asco7", "sico7", "na0-"]))].copy()

reviewer_safety_rows = [
    {
        "check_item": "automatic_google_scholar_scraping",
        "status": "PASS",
        "observed": False,
        "required": False,
        "interpretation": "Notebook generates URLs and search queries only; it does not scrape Google Scholar.",
    },
    {
        "check_item": "literature_unfriendly_queries_removed_from_manual_csv",
        "status": "PASS" if len(unfriendly_in_manual) == 0 else "CHECK",
        "observed": len(unfriendly_in_manual),
        "required": 0,
        "interpretation": "Manual CSV should avoid MP-style battery-window and condensed PCO7/CSO7/AsCO7/SiCO7 queries.",
    },
    {
        "check_item": "archived_unfriendly_query_audit_created",
        "status": "PASS" if (AUDIT_DIR / "09_literature_unfriendly_query_audit.csv").exists() else "FAIL",
        "observed": str(AUDIT_DIR / "09_literature_unfriendly_query_audit.csv"),
        "required": "audit file exists",
        "interpretation": "Problematic formulas are preserved for transparency but not used as the main manual-search workload.",
    },
    {
        "check_item": "minimum_query_coverage",
        "status": "PASS" if (query_generation_audit["n_recommended_queries"] >= 6).all() else "CHECK",
        "observed": int(query_generation_audit["n_recommended_queries"].min()),
        "required": ">=6 per candidate",
        "interpretation": "Each candidate should have enough formula, grouped-formula, variable-sodium, and phrase-based queries.",
    },
    {
        "check_item": "manual_literature_not_record_validation",
        "status": "PASS",
        "observed": "do_not_use_as_discovery_claim defaults True",
        "required": True,
        "interpretation": "Literature analogues support chemical relevance only and do not validate computed MP values/ranks.",
    },
]

reviewer_safety_df = pd.DataFrame(reviewer_safety_rows)
reviewer_safety_df.to_csv(AUDIT_DIR / "09_reviewer_safety_audit.csv", index=False)

display(reviewer_safety_df)
if len(unfriendly_in_manual):
    print("Potentially unfriendly queries still in manual CSV:")
    display(unfriendly_in_manual[["query_id", "manual_candidate_id", "search_query"]].head(30))


In [ ]:

# ============================================================
# Cell 11 — Final decision and manifest
# ============================================================

def list_output_files(base_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(base_dir.rglob("*")):
        if p.is_file():
            rows.append({
                "relative_path": str(p.relative_to(base_dir)),
                "size_bytes": p.stat().st_size,
                "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    return pd.DataFrame(rows)

required_outputs = [
    PROCESSED_DIR / "09_literature_friendly_query_table_all_alternatives.csv",
    PROCESSED_DIR / "09_literature_friendly_query_table_recommended.csv",
    PROCESSED_DIR / "09_manual_literature_annotations_BLANK_.csv",
    PROCESSED_DIR / "09_manual_literature_annotations_WITH_CARRYOVER.csv",
    AUDIT_DIR / "09_literature_unfriendly_query_audit.csv",
    AUDIT_DIR / "09_query_generation_coverage_audit.csv",
    AUDIT_DIR / "09_reviewer_safety_audit.csv",
]
missing_outputs = [str(p) for p in required_outputs if not p.exists()]
reviewer_failures = reviewer_safety_df[reviewer_safety_df["status"] == "FAIL"]
reviewer_checks = reviewer_safety_df[reviewer_safety_df["status"] == "CHECK"]

if missing_outputs or len(reviewer_failures):
    final_decision = "NO_GO_FIX_NOTEBOOK_09_OUTPUTS"
elif len(reviewer_checks):
    final_decision = "CONDITIONAL_GO_MANUAL_REVIEW_READY_CHECK_WARNINGS"
else:
    final_decision = "FULL_GO_MANUAL_REVIEW_READY_CSV"

final_info = {
    "notebook": "_CMT_PUBLIC_NOTEBOOK_FILENAME_02_",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "final_decision": final_decision,
    "input_source": input_source,
    "n_candidates": int(len(candidates_df)),
    "n_recommended_manual_queries": int(len(manual_required_df)),
    "n_archived_unfriendly_formula_rows": int(len(archived_df)),
    "missing_outputs": missing_outputs,
    "reviewer_safety_status_counts": reviewer_safety_df["status"].value_counts().to_dict(),
    "main_manual_csv_to_fill": str(PROCESSED_DIR / "09_manual_literature_annotations_WITH_CARRYOVER.csv"),
    "blank_manual_csv": str(PROCESSED_DIR / "09_manual_literature_annotations_BLANK_.csv"),
}

write_json_safe(final_info, METADATA_DIR / "09_final_decision.json")
manifest_df = list_output_files(BASE_DIR)
manifest_df.to_csv(METADATA_DIR / "09_output_manifest.csv", index=False)
save_event_log()

print("FINAL DECISION:", final_decision)
print("Main CSV to use:", final_info["main_manual_csv_to_fill"])
print("Output folder:", BASE_DIR)
display(manifest_df)


In [ ]:
from pathlib import Path
import sys
import pandas as pd


def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
p = (
    REPOSITORY_ROOT
    / "provenance"
    / "supplementary"
    / "notebook_09"
    / "manual_inputs"
    / "09_manual_literature_annotations_FILLED.csv"
)

print("Exists:", p.exists())
print("Path:", p)

df = pd.read_csv(p)
checked = df["manual_checked"].astype(str).str.lower().isin(["true", "1", "yes", "y"])

print("Rows:", len(df))
print("Checked rows:", checked.sum())
print("Candidates with checked rows:", df.loc[checked, "manual_candidate_id"].nunique())
print("Columns:", len(df.columns))
